<a href="https://colab.research.google.com/github/w-olivine29/data-analysis/blob/main/%EC%98%A4%ED%94%88%EC%86%8C%EC%8A%A4_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D_4%EA%B0%95_%EC%8B%A4%EC%8A%B5_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%88%98%EC%A7%91%20%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# 오픈소스 기반 데이터 분석 4강 - 데이터 수집


#💡 실습 전 확인

파일 업로드하기: 코드에서 파일을 읽어오는 실습이므로, 실행 전 좌측 파일 메뉴(📁)에 실습에서 제공되는 Chapter 3 파일들을 반드시 업로드하기

주의사항: 런타임 연결이 끊기거나 세션이 종료되면 가상머신이 초기화되어 파일이 삭제되므로, 재접속 시 다시 업로드하기

## 4-1 CSV 파일 읽기

In [12]:
import pandas as pd

## data.csv 파일 읽기
### 상황에 따라 옵션 값을 활용해서 읽어오면 된다
### 아래 옵션 값들은 기본값
df = pd.read_csv("data.csv", encoding = "utf-8", sep = ",", header= 0,
                 index_col=None, skiprows= None, nrows=None)


print(type(df), end = "\n\n")
print(df)

<class 'pandas.core.frame.DataFrame'>

           날짜    체중  골격근량  체지방량
0  2025.02.06  64.7  30.0  11.1
1  2025.02.04  64.0  29.3  11.6


## 4-2 JSON 파일 읽기



In [20]:
import json
import pandas as pd

## 파일을 읽고 json 라이브러리를 통해 읽어오기
## data.json 파일 출력
with open("data.json", mode = "r", encoding= "utf-8") as f :
    data = json.load(f) #읽어온 파일을 통해 json 객체생성
print(data)

## pandas 를 통해 읽어오기
## data.json 파일 DataFrame 읽기
df = pd.read_json("data.json", orient= "records", encoding="utf-8")

print(df)

{'매출데이터': [{'월': '2025-01', '매출액': 1000000, '비용': 700000, '이익': 300000}, {'월': '2025-02', '매출액': 1200000, '비용': 800000, '이익': 400000}, {'월': '2025-03', '매출액': 1500000, '비용': 900000, '이익': 600000}]}
                                               매출데이터
0  {'월': '2025-01', '매출액': 1000000, '비용': 700000,...
1  {'월': '2025-02', '매출액': 1200000, '비용': 800000,...
2  {'월': '2025-03', '매출액': 1500000, '비용': 900000,...


## 4-3 텍스트 파일 읽기 및 데이터 추출

In [21]:
import re  # re: 내장 라이브러리인 정규 표현식(Regular Expression, Regex) 모듈

## 파일(callcenter20250301.log) 오픈 및 읽기
with open('callcenter20250301.log', 'r', encoding='utf-8') as f:
    content = f.read()

## 주민등록번호 패턴 객체 생성
pattern = re.compile(r"(\d{6})-(\d{7})")

## 주민등록번호 마스킹
masked_content = pattern.sub(r"\1-*******", content)

## 마스킹된 파일(callcenter20250301_masked.log) 오픈 및 쓰기
with open("callcenter20250301_masked.log", mode = "w") as f :
    f.write(masked_content)

print("주민등록번호 마스킹 완료. 'callcenter20250301_masked.log.txt' 파일로 저장되었습니다.")

주민등록번호 마스킹 완료. 'callcenter20250301_masked.log.txt' 파일로 저장되었습니다.


## 4-4 Open-Meteo의 무료 날씨 API를 통한 특정 지역 온도 조회

In [22]:
import requests
import json

url = "https://api.open-meteo.com/v1/forecast?=&=&current=temperature_2m"
params = {
    "latitude": "37.58638333",
    "longitude": "127.0203333",
    "current": "temperature_2m"
}

try:
    ## URL 및 파라미터 전송
    response =  requests.get(url, params = params)
    response.raise_for_status()

    ## JSON 데이터 읽기
    data = response.json()

    print("API 응답:", data)
    print("서울시 종로구의 현재 온도는 : {0}{1} 입니다.".format(data['current']['temperature_2m'], data['current_units']['temperature_2m']))

except requests.exceptions.RequestException as e:
    print(f"API 호출 실패: {e}")
except json.JSONDecodeError as e:
    print(f"JSON 파싱 실패: {e}")

API 응답: {'latitude': 37.6, 'longitude': 127.0, 'generationtime_ms': 0.02372264862060547, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 29.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C'}, 'current': {'time': '2026-09-30T08:30', 'interval': 900, 'temperature_2m': 22.4}}
서울시 종로구의 현재 온도는 : 22.4°C 입니다.


## 4-5 Selenium과 lxml을 이용한 웹 스크래핑

### 크롬 다운로드

- Google Colab에는 Chrome 과 제어 드라이버가 깔려있지 않기 때문에
아래 코드를 통해 설치
    - Google Colab은 화면 디스플레이(모니터)와 기본 웹 브라우저가 없는 원격 리눅스(Ubuntu) 서버 환경

In [23]:

# 구글 공식 리눅스 배포 서버에서 우분투용 Google Chrome 설치 파일(.deb)을 다운로드
!curl -o google-chrome-stable_current_amd64.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

# 다운로드한 패키지 파일을 Colab 서버 시스템에 직접 설치
!apt install ./google-chrome-stable_current_amd64.deb -y

# 브라우저 자동 제어 라이브러리인 selenium과, 설치된 크롬 버전에 맞는 실행 드라이버를 자동으로 매칭해 주는 webdriver_manager 파이썬 패키지를 설치
!pip install selenium webdriver_manager


# Colab 서버에는 모니터 화면이 없기 때문에,
# 이후 Python 코드로 크롬 브라우저를 띄울 때는 반드시 백그라운드 구동 모드인 --headless 옵션을 추가해야 정상 작동


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  136M  100  136M    0     0   135M      0  0:00:01  0:00:01 --:--:--  135M
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1 libxtst6 session-migration
The following NEW packages will be installed:
  at-spi2-common at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1
  libxtst6 session-migration
0 upgraded, 10 newly installed, 0 to remove and 0 not upgraded.
Need to get 331 kB/143 MB of archives.
After this operation, 465 MB of addition

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from lxml import html
import time

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')               # 브라우저 창 없이 실행
chrome_options.add_argument('--no-sandbox')             # 보안모드 비활성화 (Colab 필수)
chrome_options.add_argument('--disable-dev-shm-usage')  # 메모리 부족 방지 (Colab 필수)
chrome_options.add_argument('--window-size=1920x1080')  # 창 크기 설정(가상)
chrome_options.add_argument('--disable-gpu')            # GPU 가속 비활성화 (일부 환경 안정성)
chrome_options.binary_location = "/usr/bin/google-chrome-stable"  # Colab용 크롬 경로 지정

## 드라이버 실행
driver = webdriver.Chrome(options=chrome_options)

## 사이트 접속
url = "https://professor.knou.ac.kr/jaehwachung/index.do"
driver.get(url)

## 사이트 접속 대기
time.sleep(2)

## 페이지 제목 출력
page_source = driver.page_source
tree = html.fromstring(page_source)


title_text = tree.xpath("//title/text()")
print(title_text)

## 드라이버 종료
driver.quit()

['컴퓨터과학과 정재화 교수']



# 실습 시나리오

## 공공데이터 포털 가입 및 데이터 신청

- [https://www.data.go.kr](https://www.data.go.kr)
- 한국환경공단 에어코리아 대기오염정보 데이터 신청

In [9]:
import requests

## 데이터 수집 url 및 api key 설정

url = "https://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getCtprvnRltmMesureDnsty?"
api_key = ""

params = {
    'serviceKey': api_key,
    'returnType': 'json',
    'numOfRows': '10',
    'pageNo': '1',
    'sidoName': '서울',
    'ver': '1.0'
}

## 데이터 수집
response = requests.get(url,params=params)

## 호출 성공/실패 출력
print(response.json())


{'response': {'body': {'totalCount': 40, 'items': [{'so2Grade': '1', 'coFlag': None, 'khaiValue': '76', 'so2Value': '0.003', 'coValue': '0.2', 'pm25Flag': None, 'pm10Flag': None, 'o3Grade': '2', 'pm10Value': '21', 'khaiGrade': '2', 'pm25Value': '12', 'sidoName': '서울', 'no2Flag': None, 'no2Grade': '1', 'o3Flag': None, 'pm25Grade': '2', 'so2Flag': None, 'dataTime': '2026-09-30 18:00', 'coGrade': '1', 'no2Value': '0.010', 'stationName': '중구', 'pm10Grade': '1', 'o3Value': '0.061'}, {'so2Grade': '1', 'coFlag': None, 'khaiValue': '68', 'so2Value': '0.002', 'coValue': '0.4', 'pm25Flag': None, 'pm10Flag': None, 'o3Grade': '2', 'pm10Value': '25', 'khaiGrade': '2', 'pm25Value': '7', 'sidoName': '서울', 'no2Flag': None, 'no2Grade': '1', 'o3Flag': None, 'pm25Grade': '1', 'so2Flag': None, 'dataTime': '2026-09-30 18:00', 'coGrade': '1', 'no2Value': '0.022', 'stationName': '한강대로', 'pm10Grade': '1', 'o3Value': '0.050'}, {'so2Grade': '1', 'coFlag': None, 'khaiValue': '-', 'so2Value': '0.002', 'coValue': 